# 04 ML Pipeline — Patient Readmission Prediction

In [0]:
from pyspark.sql.functions import col

gold_df = spark.read.table("healthcare_readmission.readmission.gold_patient_features")

row_count = gold_df.count()
print(f"Total rows: {row_count}")

print("\nSchema:")
gold_df.printSchema()

print("\nClass distribution (label):")
display(gold_df.groupBy("label").count().orderBy("label"))

print("\nClass distribution (readmitted):")
display(gold_df.groupBy("readmitted").count().orderBy(col("count").desc()))

Total rows: 101766

Schema:
root
 |-- features: vectorudt (nullable = true)
 |-- label: double (nullable = true)
 |-- readmitted: string (nullable = true)


Class distribution (label):


label,count
0.0,54864
1.0,35545
2.0,11357



Class distribution (readmitted):


readmitted,count
NO,54864
>30,35545
<30,11357


In [0]:
from pyspark.sql.functions import col, isnan

print("=== Null / Invalid Value Check ===\n")

null_label = gold_df.filter(col("label").isNull()).count()
nan_label = gold_df.filter(isnan(col("label"))).count()
print(f"Null labels: {null_label}")
print(f"NaN labels: {nan_label}")

null_features = gold_df.filter(col("features").isNull()).count()
print(f"Null feature vectors: {null_features}")

invalid_labels = gold_df.filter(~col("label").isin(0.0, 1.0, 2.0)).count()
print(f"Invalid label values (not in {{0, 1, 2}}): {invalid_labels}")

if null_label == 0 and nan_label == 0 and null_features == 0 and invalid_labels == 0:
    print("\nAll checks passed — no null or invalid values found.")
else:
    print("\nIssues found — review above.")

=== Null / Invalid Value Check ===

Null labels: 0
NaN labels: 0
Null feature vectors: 0
Invalid label values (not in {0, 1, 2}): 0

All checks passed — no null or invalid values found.


In [0]:
SEED = 42

train_df, test_df = gold_df.randomSplit([0.8, 0.2], seed=SEED)

train_count = train_df.count()
test_count = test_df.count()
total = train_count + test_count
print(f"Training set: {train_count} rows ({train_count / total * 100:.1f}%)")
print(f"Testing set:  {test_count} rows ({test_count / total * 100:.1f}%)")

print("\nTraining set class distribution:")
display(train_df.groupBy("label").count().orderBy("label"))

print("Testing set class distribution:")
display(test_df.groupBy("label").count().orderBy("label"))

Training set: 81225 rows (79.8%)
Testing set:  20541 rows (20.2%)

Training set class distribution:


label,count
0.0,43717
1.0,28473
2.0,9035


Testing set class distribution:


label,count
0.0,11147
1.0,7072
2.0,2322


In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    family="multinomial",
    maxIter=100,
    regParam=0.1,
)

print("Training Multinomial Logistic Regression...")
lr_model = lr.fit(train_df)
print("Logistic Regression training complete.")
print(f"Coefficient matrix: {lr_model.coefficientMatrix.numRows} classes x {lr_model.coefficientMatrix.numCols} features")

Training Multinomial Logistic Regression...
Logistic Regression training complete.
Coefficient matrix: 4 classes x 2543 features


In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=30,
    maxDepth=6,
    seed=SEED,
)

print("Training Random Forest Classifier...")
rf_model = rf.fit(train_df)
print("Random Forest training complete.")
print(f"Number of trees: {rf.getNumTrees()}")
print(f"Feature importance vector length: {rf_model.featureImportances.size}")

Training Random Forest Classifier...
Random Forest training complete.
Number of trees: <bound method _RandomForestParams.getNumTrees of RandomForestClassifier_3aca79561720>
Feature importance vector length: 2543


In [0]:
lr_predictions = lr_model.transform(test_df)
rf_predictions = rf_model.transform(test_df)

print("Logistic Regression — sample predictions:")
display(lr_predictions.select("label", "prediction", "readmitted").limit(10))

print("Random Forest — sample predictions:")
display(rf_predictions.select("label", "prediction", "readmitted").limit(10))

Logistic Regression — sample predictions:


label,prediction,readmitted
1.0,1.0,>30
0.0,0.0,NO
0.0,0.0,NO
1.0,1.0,>30
1.0,1.0,>30
1.0,0.0,>30
0.0,1.0,NO
1.0,1.0,>30
2.0,0.0,<30
0.0,1.0,NO


Random Forest — sample predictions:


label,prediction,readmitted
1.0,0.0,>30
0.0,0.0,NO
0.0,0.0,NO
1.0,0.0,>30
1.0,0.0,>30
1.0,0.0,>30
0.0,0.0,NO
1.0,0.0,>30
2.0,0.0,<30
0.0,0.0,NO


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedFMeasure")

metrics = {}

for model_name, preds in [("Logistic Regression", lr_predictions), ("Random Forest", rf_predictions)]:
    acc = evaluator_acc.evaluate(preds)
    prec = evaluator_prec.evaluate(preds)
    rec = evaluator_rec.evaluate(preds)
    f1 = evaluator_f1.evaluate(preds)

    metrics[model_name] = {
        "Accuracy": acc,
        "Weighted Precision": prec,
        "Weighted Recall": rec,
        "Weighted F1": f1,
    }

    print(f"\n{model_name} Evaluation:")
    print(f"  Accuracy:           {acc:.4f}")
    print(f"  Weighted Precision: {prec:.4f}")
    print(f"  Weighted Recall:    {rec:.4f}")
    print(f"  Weighted F1:        {f1:.4f}")


Logistic Regression Evaluation:
  Accuracy:           0.5851
  Weighted Precision: 0.5470
  Weighted Recall:    0.5851
  Weighted F1:        0.5327

Random Forest Evaluation:
  Accuracy:           0.5427
  Weighted Precision: 0.2945
  Weighted Recall:    0.5427
  Weighted F1:        0.3818


In [0]:
print("Logistic Regression — Confusion Matrix")
print("Rows: Actual Label | Columns: Predicted Label\n")
lr_cm = lr_predictions.groupBy("label").pivot("prediction").count().na.fill(0).orderBy("label")
display(lr_cm)

print("\nRandom Forest — Confusion Matrix")
print("Rows: Actual Label | Columns: Predicted Label\n")
rf_cm = rf_predictions.groupBy("label").pivot("prediction").count().na.fill(0).orderBy("label")
display(rf_cm)

Logistic Regression — Confusion Matrix
Rows: Actual Label | Columns: Predicted Label



label,0.0,1.0,2.0
0.0,9493,1626,28
1.0,4546,2490,36
2.0,1491,795,36



Random Forest — Confusion Matrix
Rows: Actual Label | Columns: Predicted Label



label,0.0,1.0
0.0,11147,0
1.0,7072,0
2.0,2321,1


In [0]:
comparison_rows = [
    (model_name, m["Accuracy"], m["Weighted Precision"], m["Weighted Recall"], m["Weighted F1"])
    for model_name, m in metrics.items()
]

comparison_df = spark.createDataFrame(
    comparison_rows,
    ["Model", "Accuracy", "Weighted Precision", "Weighted Recall", "Weighted F1"],
)

print("Model Comparison Table:")
display(comparison_df)

Model Comparison Table:


Model,Accuracy,Weighted Precision,Weighted Recall,Weighted F1
Logistic Regression,0.5851224380507278,0.5469748908754628,0.5851224380507278,0.5326606329922223
Random Forest,0.5426707560488778,0.2945058869365551,0.5426707560488778,0.38180647695754355


In [0]:
from pyspark.sql.functions import col

best_model_name = max(metrics, key=lambda k: metrics[k]["Weighted F1"])
print(f"Selected model (highest Weighted F1): {best_model_name}")
print(f"  Weighted F1: {metrics[best_model_name]['Weighted F1']:.4f}")

if best_model_name == "Logistic Regression":
    best_model = lr_model
    final_predictions = lr_predictions
else:
    best_model = rf_model
    final_predictions = rf_predictions

print("\nSample of final predictions:")
display(
    final_predictions.select(
        col("label").alias("actual_label"),
        col("prediction").alias("predicted_label"),
        col("readmitted"),
    ).limit(20)
)

Selected model (highest Weighted F1): Logistic Regression
  Weighted F1: 0.5327

Sample of final predictions:


actual_label,predicted_label,readmitted
1.0,1.0,>30
0.0,0.0,NO
0.0,0.0,NO
1.0,1.0,>30
1.0,1.0,>30
1.0,0.0,>30
0.0,1.0,NO
1.0,1.0,>30
2.0,0.0,<30
0.0,1.0,NO


In [0]:
from pyspark.sql.functions import col

predictions_output = final_predictions.select(
    col("label").alias("actual_label"),
    col("prediction").alias("predicted_label"),
    col("readmitted"),
)

predictions_output.write.mode("overwrite").saveAsTable("healthcare_readmission.readmission.readmission_predictions")

written_count = predictions_output.count()
print(f"Predictions saved to healthcare_readmission.readmission.readmission_predictions")
print(f"Rows written: {written_count}")

Predictions saved to healthcare_readmission.readmission.readmission_predictions
Rows written: 20541


In [0]:
verify_df = spark.read.table("healthcare_readmission.readmission.readmission_predictions")

print(f"Prediction table row count: {verify_df.count()}")
print("\nSchema:")
verify_df.printSchema()
print("\nSample rows:")
display(verify_df.limit(10))

Prediction table row count: 20541

Schema:
root
 |-- actual_label: double (nullable = true)
 |-- predicted_label: double (nullable = true)
 |-- readmitted: string (nullable = true)


Sample rows:


actual_label,predicted_label,readmitted
1.0,1.0,>30
0.0,0.0,NO
0.0,0.0,NO
1.0,1.0,>30
1.0,1.0,>30
1.0,0.0,>30
0.0,1.0,NO
1.0,1.0,>30
2.0,0.0,<30
0.0,1.0,NO
